In [3]:
import pandas as pd
import torch
import pickle
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from transformers import AutoTokenizer, AutoModel



/Users/samirkadariya/opt/anaconda3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
df = pd.read_pickle(
    "./1789_2023_MAPPINGS_N_N1.pkl"
)

In [9]:
def setup_model_and_tokenizer():
    token = 'REDACTED_HF_TOKEN'
    model_name = 'sentence-transformers/all-mpnet-base-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
    model = AutoModel.from_pretrained(model_name, token=token)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    return tokenizer, model, device


def encode_text(texts, tokenizer, model, device, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        with torch.no_grad():
            model_output = model(**encoded_input)
        token_embeddings = model_output.last_hidden_state
        attention_mask = encoded_input['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

# Function to calculate cosine similarity between two sets of embeddings
def calculate_cosine_similarity(embeddings1, embeddings2):
    return torch.nn.functional.cosine_similarity(embeddings1, embeddings2, dim=1)

In [11]:
def setup_model_and_tokenizer():
    token = 'REDACTED_HF_TOKEN'
    model_name = 'sentence-transformers/all-mpnet-base-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
    model = AutoModel.from_pretrained(model_name, token=token)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    return tokenizer, model, device


def encode_text(texts, tokenizer, model, device, batch_size=64):
    all_embeddings = []
    # Add tqdm for batch processing
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing batches"):
        batch = texts[i:i+batch_size]
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        with torch.no_grad():
            model_output = model(**encoded_input)
        token_embeddings = model_output.last_hidden_state
        attention_mask = encoded_input['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)


def encode_text_batch_parallel(batch_data, tokenizer, model, device):
    texts, batch_idx = batch_data
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        model_output = model(**encoded_input)
    token_embeddings = model_output.last_hidden_state
    attention_mask = encoded_input['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    embeddings = sum_embeddings / sum_mask
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu(), batch_idx


def encode_text_parallel(texts, tokenizer, model, device, batch_size=64, max_workers=4):
    # Prepare batches
    batches = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batches.append((batch_texts, i))
    
    # Process batches in parallel
    all_embeddings = torch.zeros((len(texts), model.config.hidden_size))
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(encode_text_batch_parallel, batch, tokenizer, model, device): idx 
            for idx, batch in enumerate(batches)
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing batches in parallel"):
            embeddings, start_idx = future.result()
            end_idx = min(start_idx + batch_size, len(texts))
            all_embeddings[start_idx:end_idx] = embeddings
            
    return all_embeddings


# Function to calculate cosine similarity between two sets of embeddings
def calculate_cosine_similarity(embeddings1, embeddings2):
    return torch.nn.functional.cosine_similarity(embeddings1, embeddings2, dim=1)


In [12]:
# Main processing
print("Loading data...")
df = pd.read_pickle("./1789_2023_MAPPINGS_N_N1.pkl")

print("Setting up model and tokenizer...")
tokenizer, model, device = setup_model_and_tokenizer()

# Prepare texts for embedding
print("Preparing texts...")
descriptions_n = df['Description_N'].fillna("").astype(str).tolist()
descriptions_2023 = df['Mapped_2023_Description'].fillna("").astype(str).tolist()

# Determine optimal number of workers based on system
max_workers = min(8, (os.cpu_count() or 1))
print(f"Using {max_workers} parallel workers")

# Create embeddings using parallelization
print("Creating embeddings for Description_N...")
embeddings_n = encode_text_parallel(descriptions_n, tokenizer, model, device, max_workers=max_workers)

print("Creating embeddings for Mapped_2023_Description...")
embeddings_2023 = encode_text_parallel(descriptions_2023, tokenizer, model, device, max_workers=max_workers)

# Calculate cosine similarity
print("Calculating cosine similarity...")
cosine_similarities = calculate_cosine_similarity(embeddings_n, embeddings_2023)


Loading data...
Setting up model and tokenizer...
Preparing texts...
Using 8 parallel workers
Creating embeddings for Description_N...


Processing batches in parallel:   0%|          | 0/11645 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Processing batches in parallel: 100%|██████████| 11645/11645 [2:46:42<00:00,  1.16it/s]  


Creating embeddings for Mapped_2023_Description...


Processing batches in parallel: 100%|██████████| 11645/11645 [1:30:39<00:00,  2.14it/s] 


Calculating cosine similarity...


In [13]:

# Add to dataframe
df['embedding_similarity'] = cosine_similarities.numpy()

# Save the results

In [15]:
df.to_pickle("./1789_2023_MAPPINGS_N_N1_with_similarity.pkl")


In [16]:
df

,from_year,to_year,year_N,year_N1,HS_N,HS_N1,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,Description_N,Description_N1,desc_N,desc_N1,Mapped_2023_HS,Mapped_2023_Description,embedding_similarity
0,2022,2023,2022,2023,01023100,01023100,1.000000,1.000000,1.000000,1.000000,Live purebred breeding buffalo,Live purebred breeding buffalo,live purebred breeding buffalo,live purebred breeding buffalo,01023100,Live purebred breeding buffalo,1.000000
1,2022,2023,2022,2023,01019040,01019040,1.000000,1.000000,1.000000,1.000000,Mules and hinnies not imported for immediate s...,Mules and hinnies not imported for immediate s...,mules and hinnies not imported for immediate s...,mules and hinnies not imported for immediate s...,01019040,Mules and hinnies not imported for immediate s...,1.000000
2,2022,2023,2022,2023,01031000,01031000,1.000000,1.000000,1.000000,1.000000,Live purebred breeding swine,Live purebred breeding swine,live purebred breeding swine,live purebred breeding swine,01031000,Live purebred breeding swine,1.000000
3,2022,2023,2022,2023,01022100,01022100,1.000000,1.000000,1.000000,1.000000,Live purebred breeding cattle,Live purebred breeding cattle,live purebred breeding cattle,live purebred breeding cattle,01022100,Live purebred breeding cattle,1.000000
4,2022,2023,2022,2023,01012900,01012900,1.000000,1.000000,1.000000,1.000000,Live horses other than purebred breeding horses,Live horses other than purebred breeding horses,live horses other than purebred breeding horses,live horses other than purebred breeding horses,01012900,Live horses other than purebred breeding horses,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745501,1789,1790,1789,1790,1789_56_canes_walking_sticks_and_whips,1790_129_canes_walking_sticks_and_whips,1.000000,1.000000,1.000000,1.000000,"canes, walking sticks and whips","canes, walking sticks and whips","canes, walking sticks and whips","canes, walking sticks and whips",32041940,Synthetic organic coloring matter and preparat...,0.094847
745502,1789,1790,1789,1790,1789_61_playing_cards,1790_110_playing_cards,1.000000,1.000000,1.000000,1.000000,playing cards,playing cards,playing cards,playing cards,20089929,"Grapes, otherwise prepared or preserved, nesoi",0.035573
745503,1789,1790,1789,1790,1789_62_every_coach_chariot_or_other_four_wheel,1790_138_all_coaches_chariots_phaetons_chaises_ch,0.836723,0.120000,0.594340,0.669140,"every coach, chariot or other four wheel carri...","All coaches, chariots, phaetons, chaises, chai...","every coach, chariot or other four wheel carri...","all coaches, chariots, phaetons, chaises, chai...",32041940,Synthetic organic coloring matter and preparat...,0.206852
745504,1789,1790,1789,1790,1789_58_brushes,1790_131_brushes,1.000000,1.000000,1.000000,1.000000,brushes,brushes,brushes,brushes,63025930,"Table linen, of textile materials other than o...",0.215003
